In [16]:
"""***WARNING THIS IS QUITE CLUTTERED, HARD TO UNDERSTAND, AND BLOATED!! (SHOULD BE FIXED IN FUTURE)
HOW TO RUN:
everything in this code should be runnable as long as the board_list.py is in the same folder as this file.
feel free to run each cell one by one or run all"""
import random
from collections import defaultdict
import math
#^from previous model^
import time #time our runs
from collections import deque #for AC-3
from boards_list import boards_easy, boards_medium, boards_hard #grabs the boards from boards.py


# **Backtracking search algorithm (From Model 1)**

In [17]:
def find_empty(board): #checks for if the number is a zero (empty)
  for i in range(len(board)):
    for j in range(len(board[0])):
      if board[i][j] == 0:
        return (i, j)
  return None

def valid(board, num, pos): #checks for valid numbers within the row, col, and box
  for i in range(len(board[0])):
    if board[pos[0]][i] == num and pos[1] != i:
      return False
  for j in range(len(board)):
    if board[j][pos[1]] == num and pos[0] != j:
      return False
  box_row = 3*(pos[0]//3)
  box_col = 3*(pos[1]//3)
  box_end_row = box_row + 3
  box_end_col = box_col + 3

  for i in range(box_row, box_end_row):
    for j in range(box_col, box_end_col):
      if board[i][j] == num and (i, j) != pos:
        return False
  return True

def solution(board):
  if len(board) != 9:
    return False
  for row in board:
    if len(row) != 9:
      return False

  find = find_empty(board)
  if find is not None:
    return False

  for i in range(len(board)):
    for j in range(len(board[0])):
      if not valid(board, board[i][j], (i,j)):
        return False
      if not (1<= board[i][j]<=9):
        return False
  return True

def backtrack(board):
    find = find_empty(board)
    if find is None:
        return board
    else:
        row, col = find
    for i in range(1, 10):
      if valid(board, i, (row, col)):
          board[row][col] = i
          result = backtrack(board)

          if result is not None:
            return result

          board[row][col] = 0
    return None



# **Model Two Building**

**split into parts for differing tasks in solving the board, ex: (the 1-9 3x3 mini-regions), forward checking, AC-3, and MRV**

In [18]:
#row col celling
cells = [(row, col) for row in range(9) for col in range(9)]

#looking at the rows and cols from a cell and the 3x3 on top of it.
def loc_regionals():
    region = {}
    for (row, col) in cells:
        loc = set()

        #row and col same check
        for i in range(9):
            loc.add((row, i)) #rowmatch
            loc.add((i, col)) #colmatch

        #3x3s
        minirow, minicol = 3 * (row // 3), 3 * (col // 3)
        for i in range(minirow, minirow + 3):
            for j in range(minicol, minicol + 3):
                loc.add((i, j))
        #dont count the cell itself
        loc.discard((row, col))
        region[(row, col)] = loc


    return region


#The cells by its 20 peers
regionals = loc_regionals()


"""setup our domains for the cells we got using the clues, and possible digits in zeroes
(just a start for the actual additions forward/AC3) """
def domaining(board):
    return {(row, col): ({board[row][col]} if board[row][col] else set(range(1, 10))) for (row, col ) in cells}

# **Putting in Forward Checking and AC-3**

In [19]:
"""Forward Checking implementation
 We place a value ('val') in every single cell variable ('var')
 No variables in region can be the value of the current cell."""
def fwd_check(domains, var, val):
    changes = []
    for loc in regionals[var]:
        if val in domains[loc]:
            if len(domains[loc]) == 1:
                #if this happens the only value left in the region is the val, but its taken by the main cell already.
                return None
            domains[loc].discard(val) #kill unusable values (current 'val')
            changes.append(loc) #keep the changes remembered
    return changes

"""Helper for AC-3 main function later, it contrls the constraints for xi and xj (supposed to be xi != xj)
domain values are capable iff xj has a different value from x.
This returns True if xi has a value removed."""
def xConstraint(domains, xi, xj):
    domainj = domains[xj]
    if len(domainj) == 1:
        (v,) = domainj
        if v in domains[xi]:
            domains[xi].discard(v)
            return True
    return False


"""Arc consistency function, runs using the xi, xj from previous function
for queuing arcs and detecting whether or not the domain of xi shrinks, and
then re checking neighbors of xk and xj when more restrictions take place."""
def arcConsist(domains, queue=None):
    if queue is None: #Do ordered pair for the entirety of the arc (81 cells and their 20 peers -> 81 x 20 = 1620 arcs)
        #queue is meant to be starting arcs,
        queue = deque((xi, xj) for xi in cells for xj in regionals[xi])

    while queue:
        xi, xj = queue.popleft()

        if xConstraint(domains, xi, xj):
            if not domains[xi]:
                return False

            for xk in regionals[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True



# **Implementing the search**

In [20]:
""""""

def solve(board, mrv=True, fc=True, AC3=True):
    trackStats = {"Nodes": 0, "Backtracks": 0} #measure info of search
    board = [row[:] for row in board] #dont modify the original boards, make your own
    useDomains = fc or AC3 #dont worry about domains unless Forward Checking or Arc Consistency 3 is active.

    def candidates(domains, loc):
        if useDomains:
            return domains[loc] #use respective strat

        return {v for v in range(1, 10) if valid(board, v, loc)}


    def search(domains):
        trackStats["Nodes"] += 1 #count the cells > node of search tree

        noval = [col for col in cells if board[col[0]][col[1]] == 0] #gather unassigned region
        if not noval:
            return True # if there isn't an empty cell/region then its great!


        """MRV implementation. The minimum remaining value heuristic makes it so the cell that has the least amount of possible values left is the one chosen in order to find failures quicker. mrvCalc checks cell for candidates per cell"""
        def mrvCalc(cell):
            candidateCount = len(candidates(domains, cell))
            emptyPeers = sum(1 for loc in regionals[cell] if board[loc[0]][loc[1]] == 0)
            return (candidateCount, -emptyPeers)


        if mrv:
             var = min(noval, key=mrvCalc)
        else:
            var = noval[0] #no mrv, just first cell

        row, col = var

        #candidate testing
        for val in sorted(candidates(domains, var)):
            new = None #branching domains placeholder

            if useDomains:  #copy the domains in case of pruning
                new = {k: set(v) for k, v in domains.items()}
                new[var] = {val} #domain = val
                setting = True #ph
                changed = [loc for loc in regionals[var]]

                #
                if fc:
                    #forwarc checking engaged in solve
                    changed = fwd_check(new, var, val)
                    setting = changed is not None # no more available choices


                """forward checking removed the value from the region so when the arcs
                find nothing they don't have to do anything else. They will instead point to
                forward checked changed peers in region."""
                if setting and AC3:
                    if fc:
                        queue = deque((quepos, loc) for loc in changed for quepos in regionals[loc])
                    else:
                        #Triggers when there is no forward checking, but AC-3 does it itself normally
                        queue = deque((loc, var) for loc in regionals[var])
                    setting = arcConsist(new, queue)

                #Propagation has shown our value isn't going to work, move on
                #without recursing more.

                if not setting:
                    trackStats["Backtracks"] += 1
                    continue

            #set value, keep going
            board[row][col] = val
            if search(new):
                return True #found solution

            #search failure, undo val
            board[row][col] = 0
            trackStats["Backtracks"] += 1
        return False #all values didn't work, backtrack required.


        #configs before actually searching
    domains = None
    if useDomains:
        domains = domaining(board) #empties 1-9, gives v

        #run arc consistency on board, if it fails, unsolvable.
        if AC3:
            if not arcConsist(domains):
                return None, trackStats
        #Forward Checking only active, glives clues so empty cells domains are shown in starting nums
        else:
            for (row, col) in cells:
                if board[row][col] and fwd_check(domains, (row, col), board[row][col]) is None:
                    return None, trackStats #conflict > unsolvable
    return (board, trackStats) if search(domains) else (None, trackStats)



# **Ant Colony Optimization Solver**


In [21]:
def solve_aco(
    board,
    num_ants=10,
    max_iters=200,
    rho=0.9,       # Pheromone persistence (1 - evaporation rate)
    q0=0.9,        # Exploitation vs. exploration threshold
    evap_best=0.005 # Best-value pheromone evaporation
):
    stats = {"Iterations": 0, "AntsRun": 0, "BestRemaining": 81}
    base_board = [row[:] for row in board]

    # 1. Initial constraint propagation using your existing AC-3
    base_domains = domaining(base_board)
    if not arcConsist(base_domains):
        return None, stats

    # If AC-3 alone solved the puzzle, return immediately
    if all(len(base_domains[c]) == 1 for c in cells):
        solved = [[next(iter(base_domains[(r, c)])) for c in range(9)] for r in range(9)]
        stats["BestRemaining"] = 0
        return solved, stats

    # 2. Initialize pheromone matrix: tau[row][col][val_idx]
    tau_0 = 1.0 / 9.0
    tau = [[[tau_0 for _ in range(9)] for _ in range(9)] for _ in range(9)]

    best_domains = None
    best_fixed_count = 0
    best_pheromone_reward = 0.0

    for iteration in range(1, max_iters + 1):
        stats["Iterations"] = iteration
        iter_best_domains = None
        iter_best_fixed = -1

        for _ in range(num_ants):
            stats["AntsRun"] += 1
            # Copy the initial AC-3 pruned domains for this ant
            ant_domains = {k: set(v) for k, v in base_domains.items()}

            # Track cells that still need a decision
            unassigned = [c for c in cells if len(ant_domains[c]) > 1]
            # Shuffle order slightly so ties in MRV are broken randomly by ants
            random.shuffle(unassigned)

            failed = False
            while unassigned:
                # Pick cell with Minimum Remaining Values (MRV)
                var = min(unassigned, key=lambda c: len(ant_domains[c]))
                unassigned.remove(var)

                candidates = list(ant_domains[var])
                if not candidates:
                    failed = True
                    break
                if len(candidates) == 1:
                    # Already forced by previous propagation
                    continue

                r, c = var
                # 3. Ant chooses a value using pheromones (ACS rule)
                if random.random() < q0:
                    # Greedy choice: highest pheromone
                    val = max(candidates, key=lambda v: tau[r][c][v - 1])
                else:
                    # Roulette wheel (probabilistic) choice
                    weights = [tau[r][c][v - 1] for v in candidates]
                    val = random.choices(candidates, weights=weights, k=1)[0]

                # Assign value and propagate constraints via AC-3
                ant_domains[var] = {val}
                queue = deque((peer, var) for peer in regionals[var])
                if not arcConsist(ant_domains, queue):
                    failed = True
                    break

                # Remove any other cells that just got forced to 1 value
                unassigned = [cell for cell in unassigned if len(ant_domains[cell]) > 1]

            # Count how many cells this ant successfully locked in
            if not failed:
                fixed_count = sum(1 for c in cells if len(ant_domains[c]) == 1)
            else:
                # Partial progress before hitting a dead end
                fixed_count = sum(1 for c in cells if len(ant_domains[c]) == 1)

            if fixed_count > iter_best_fixed:
                iter_best_fixed = fixed_count
                iter_best_domains = ant_domains

            # Check if this ant completely solved the board
            if not failed and fixed_count == 81:
                solved_board = [
                    [next(iter(ant_domains[(r, c)])) for c in range(9)]
                    for r in range(9)
                ]
                stats["BestRemaining"] = 0
                return solved_board, stats

        # 4. Global Pheromone Update (reward the iteration's best ant)
        if iter_best_fixed > best_fixed_count:
            best_fixed_count = iter_best_fixed
            best_domains = iter_best_domains
            stats["BestRemaining"] = 81 - best_fixed_count

        # Pheromone reward is inversely proportional to unplaced cells
        remaining = 81 - iter_best_fixed
        delta_tau = 81.0 / (remaining if remaining > 0 else 1)

        if delta_tau > best_pheromone_reward:
            best_pheromone_reward = delta_tau
            best_domains = iter_best_domains

        # Evaporate and deposit along the best ant's assignments
        if best_domains is not None:
            for (r, c) in cells:
                if len(best_domains[(r, c)]) == 1:
                    chosen_val = next(iter(best_domains[(r, c)]))
                    v_idx = chosen_val - 1
                    tau[r][c][v_idx] = (
                        rho * tau[r][c][v_idx] + (1.0 - rho) * best_pheromone_reward
                    )

        # Evaporate best pheromone value slightly to encourage exploration if stuck
        best_pheromone_reward *= (1.0 - evap_best)

    return None, stats

In [22]:
def runTestsACO(board, num_ants=10, max_iters=200):
    startTime = time.perf_counter()
    result, stats = solve_aco(board, num_ants=num_ants, max_iters=max_iters)
    elapsed = time.perf_counter() - startTime

    if result is not None:
        for row in result:
            print(row)
        print(f"Valid solution: {solution(result)}, {stats} in {elapsed:.4f} seconds\n")
    else:
        print(f"No solution found. {stats} in {elapsed:.4f} seconds\n")

# Run on your test boards
print("EASY BOARDS (ACO)")
for i in range(len(boards_easy)):
    print(f"Board {i}:")
    runTestsACO(boards_easy[i])

print("MEDIUM BOARDS (ACO)")
for i in range(len(boards_medium)):
    print(f"Board {i + 10}:")
    runTestsACO(boards_medium[i])

print("HARD BOARDS (ACO)")
for i in range(len(boards_hard)):
    print(f"Board {i + 20}:")
    runTestsACO(boards_hard[i])

EASY BOARDS (ACO)
Board 0:
[9, 6, 1, 4, 8, 2, 5, 3, 7]
[7, 4, 5, 3, 1, 9, 2, 8, 6]
[8, 3, 2, 7, 5, 6, 1, 4, 9]
[5, 9, 3, 1, 7, 4, 6, 2, 8]
[2, 7, 6, 9, 3, 8, 4, 1, 5]
[1, 8, 4, 6, 2, 5, 9, 7, 3]
[6, 1, 9, 8, 4, 7, 3, 5, 2]
[3, 2, 8, 5, 6, 1, 7, 9, 4]
[4, 5, 7, 2, 9, 3, 8, 6, 1]
Valid solution: True, {'Iterations': 0, 'AntsRun': 0, 'BestRemaining': 0} in 0.0059 seconds

Board 1:
[8, 2, 6, 7, 4, 5, 9, 1, 3]
[7, 5, 3, 1, 9, 2, 4, 6, 8]
[9, 1, 4, 6, 3, 8, 7, 2, 5]
[6, 4, 9, 5, 2, 1, 8, 3, 7]
[5, 8, 7, 3, 6, 9, 2, 4, 1]
[1, 3, 2, 4, 8, 7, 5, 9, 6]
[3, 7, 5, 2, 1, 4, 6, 8, 9]
[2, 6, 8, 9, 7, 3, 1, 5, 4]
[4, 9, 1, 8, 5, 6, 3, 7, 2]
Valid solution: True, {'Iterations': 0, 'AntsRun': 0, 'BestRemaining': 0} in 0.0057 seconds

Board 2:
[9, 4, 6, 8, 1, 5, 2, 7, 3]
[3, 5, 1, 2, 7, 9, 8, 6, 4]
[8, 7, 2, 6, 3, 4, 9, 5, 1]
[1, 6, 7, 9, 8, 3, 5, 4, 2]
[5, 3, 8, 4, 2, 7, 6, 1, 9]
[4, 2, 9, 5, 6, 1, 7, 3, 8]
[2, 1, 5, 3, 9, 6, 4, 8, 7]
[6, 9, 3, 7, 4, 8, 1, 2, 5]
[7, 8, 4, 1, 5, 2, 3, 9, 6]
Valid solutio

Hard coded test boards (from Model 1)

In [23]:
test_board_easy = [
    [4,0,3,5,2,0,0,7,6],
    [0,1,8,0,6,0,0,0,2],
    [0,0,6,4,0,8,0,9,0],
    [0,0,2,8,1,0,0,6,0],
    [0,3,5,0,0,0,4,0,1],
    [1,0,0,3,4,6,5,0,0],
    [0,0,0,6,8,9,0,0,4],
    [0,6,1,0,0,0,7,3,0],
    [8,2,0,0,0,3,0,0,0]
]

test_board_medium = [
    [0,0,0,4,0,0,5,0,0],
    [0,0,0,8,0,0,0,0,0],
    [0,6,0,0,0,0,9,0,0],
    [7,0,3,0,4,0,0,2,0],
    [0,2,0,6,0,9,3,4,0],
    [0,9,0,0,0,2,1,0,0],
    [0,7,0,0,0,3,0,1,0],
    [2,0,0,1,6,0,0,0,0],
    [0,0,6,0,0,8,0,3,0]
]

test_board_hard = [
    [0,2,9,0,4,0,0,8,0],
    [0,0,0,0,2,1,0,4,0],
    [0,0,0,8,3,9,0,0,0],
    [0,0,0,0,9,0,0,0,1],
    [3,0,0,6,0,5,0,0,8],
    [2,6,0,0,0,0,0,3,0],
    [4,0,0,0,0,0,7,1,0],
    [0,1,0,0,0,8,0,0,6],
    [0,0,0,0,0,0,0,0,0]
]

In [24]:
#Simple function for solving the boards
def runTests(board, mrv=True, fc=True, AC3=True):
    startTime = time.perf_counter()
    result, stats = solve(board, mrv=mrv, fc=fc, AC3=AC3)
    if result is not None:
        for row in result:
            print(row)
        endTime = time.perf_counter()
        elapsed = endTime - startTime
        print(f'Valid solution: {solution(result)}, {stats} in {elapsed:.4f} seconds \n MRV = {mrv}, Forward Checking = {fc}, Arc Consistnecy 3 = {AC3} \n')
    else:
        print("No solution found.")

Run solver function

In [25]:
for i in range(len(boards_easy)):
    runTests(boards_easy[i])

[9, 6, 1, 4, 8, 2, 5, 3, 7]
[7, 4, 5, 3, 1, 9, 2, 8, 6]
[8, 3, 2, 7, 5, 6, 1, 4, 9]
[5, 9, 3, 1, 7, 4, 6, 2, 8]
[2, 7, 6, 9, 3, 8, 4, 1, 5]
[1, 8, 4, 6, 2, 5, 9, 7, 3]
[6, 1, 9, 8, 4, 7, 3, 5, 2]
[3, 2, 8, 5, 6, 1, 7, 9, 4]
[4, 5, 7, 2, 9, 3, 8, 6, 1]
Valid solution: True, {'Nodes': 47, 'Backtracks': 0} in 0.0108 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[8, 2, 6, 7, 4, 5, 9, 1, 3]
[7, 5, 3, 1, 9, 2, 4, 6, 8]
[9, 1, 4, 6, 3, 8, 7, 2, 5]
[6, 4, 9, 5, 2, 1, 8, 3, 7]
[5, 8, 7, 3, 6, 9, 2, 4, 1]
[1, 3, 2, 4, 8, 7, 5, 9, 6]
[3, 7, 5, 2, 1, 4, 6, 8, 9]
[2, 6, 8, 9, 7, 3, 1, 5, 4]
[4, 9, 1, 8, 5, 6, 3, 7, 2]
Valid solution: True, {'Nodes': 46, 'Backtracks': 0} in 0.0089 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[9, 4, 6, 8, 1, 5, 2, 7, 3]
[3, 5, 1, 2, 7, 9, 8, 6, 4]
[8, 7, 2, 6, 3, 4, 9, 5, 1]
[1, 6, 7, 9, 8, 3, 5, 4, 2]
[5, 3, 8, 4, 2, 7, 6, 1, 9]
[4, 2, 9, 5, 6, 1, 7, 3, 8]
[2, 1, 5, 3, 9, 6, 4, 8, 7]
[6, 9, 3, 7, 4, 8, 1, 2, 5

In [26]:
for i in range(len(boards_medium)):
    runTests(boards_medium[i])

[1, 6, 2, 7, 4, 9, 8, 5, 3]
[5, 7, 4, 3, 8, 2, 1, 9, 6]
[8, 9, 3, 5, 6, 1, 7, 2, 4]
[9, 4, 7, 8, 2, 6, 5, 3, 1]
[3, 5, 1, 9, 7, 4, 2, 6, 8]
[6, 2, 8, 1, 3, 5, 4, 7, 9]
[4, 1, 5, 6, 9, 7, 3, 8, 2]
[7, 8, 9, 2, 1, 3, 6, 4, 5]
[2, 3, 6, 4, 5, 8, 9, 1, 7]
Valid solution: True, {'Nodes': 53, 'Backtracks': 4} in 0.0118 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[5, 6, 8, 7, 4, 9, 2, 1, 3]
[3, 1, 9, 5, 2, 6, 7, 4, 8]
[7, 4, 2, 3, 8, 1, 6, 9, 5]
[9, 8, 1, 6, 5, 2, 4, 3, 7]
[2, 5, 4, 9, 3, 7, 1, 8, 6]
[6, 7, 3, 4, 1, 8, 9, 5, 2]
[1, 9, 7, 8, 6, 3, 5, 2, 4]
[4, 3, 6, 2, 9, 5, 8, 7, 1]
[8, 2, 5, 1, 7, 4, 3, 6, 9]
Valid solution: True, {'Nodes': 52, 'Backtracks': 2} in 0.0130 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[2, 7, 4, 8, 1, 6, 3, 9, 5]
[3, 9, 5, 7, 2, 4, 8, 6, 1]
[8, 1, 6, 9, 3, 5, 7, 2, 4]
[6, 4, 8, 3, 7, 1, 9, 5, 2]
[7, 5, 3, 6, 9, 2, 1, 4, 8]
[1, 2, 9, 5, 4, 8, 6, 7, 3]
[4, 3, 2, 1, 6, 9, 5, 8, 7]
[5, 6, 7, 4, 8, 3, 2, 1, 9

In [27]:
for i in range(len(boards_hard)):
    runTests(boards_hard[i])

[9, 1, 7, 6, 3, 2, 8, 4, 5]
[6, 4, 2, 5, 8, 7, 1, 9, 3]
[8, 3, 5, 9, 1, 4, 2, 7, 6]
[5, 6, 3, 4, 7, 1, 9, 8, 2]
[7, 2, 9, 8, 6, 5, 3, 1, 4]
[1, 8, 4, 2, 9, 3, 6, 5, 7]
[4, 9, 8, 7, 2, 6, 5, 3, 1]
[3, 7, 6, 1, 5, 9, 4, 2, 8]
[2, 5, 1, 3, 4, 8, 7, 6, 9]
Valid solution: True, {'Nodes': 55, 'Backtracks': 3} in 0.0161 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[4, 7, 1, 3, 9, 5, 2, 6, 8]
[2, 9, 5, 8, 7, 6, 4, 3, 1]
[3, 8, 6, 1, 4, 2, 7, 9, 5]
[1, 4, 8, 9, 2, 7, 6, 5, 3]
[6, 2, 3, 4, 5, 8, 1, 7, 9]
[7, 5, 9, 6, 1, 3, 8, 4, 2]
[9, 3, 2, 7, 8, 4, 5, 1, 6]
[5, 1, 7, 2, 6, 9, 3, 8, 4]
[8, 6, 4, 5, 3, 1, 9, 2, 7]
Valid solution: True, {'Nodes': 97, 'Backtracks': 55} in 0.0199 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[7, 5, 8, 4, 2, 6, 1, 3, 9]
[2, 9, 3, 5, 1, 7, 8, 6, 4]
[6, 1, 4, 8, 3, 9, 5, 7, 2]
[8, 4, 2, 6, 7, 3, 9, 5, 1]
[5, 7, 1, 9, 4, 8, 3, 2, 6]
[9, 3, 6, 2, 5, 1, 4, 8, 7]
[1, 8, 9, 3, 6, 2, 7, 4, 5]
[3, 2, 5, 7, 9, 4, 6, 1, 

In [28]:
for i in range(len(boards_easy)):
    for j in range(len(boards_easy)):
        runTests(boards_easy[j])

[9, 6, 1, 4, 8, 2, 5, 3, 7]
[7, 4, 5, 3, 1, 9, 2, 8, 6]
[8, 3, 2, 7, 5, 6, 1, 4, 9]
[5, 9, 3, 1, 7, 4, 6, 2, 8]
[2, 7, 6, 9, 3, 8, 4, 1, 5]
[1, 8, 4, 6, 2, 5, 9, 7, 3]
[6, 1, 9, 8, 4, 7, 3, 5, 2]
[3, 2, 8, 5, 6, 1, 7, 9, 4]
[4, 5, 7, 2, 9, 3, 8, 6, 1]
Valid solution: True, {'Nodes': 47, 'Backtracks': 0} in 0.0112 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[8, 2, 6, 7, 4, 5, 9, 1, 3]
[7, 5, 3, 1, 9, 2, 4, 6, 8]
[9, 1, 4, 6, 3, 8, 7, 2, 5]
[6, 4, 9, 5, 2, 1, 8, 3, 7]
[5, 8, 7, 3, 6, 9, 2, 4, 1]
[1, 3, 2, 4, 8, 7, 5, 9, 6]
[3, 7, 5, 2, 1, 4, 6, 8, 9]
[2, 6, 8, 9, 7, 3, 1, 5, 4]
[4, 9, 1, 8, 5, 6, 3, 7, 2]
Valid solution: True, {'Nodes': 46, 'Backtracks': 0} in 0.0091 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[9, 4, 6, 8, 1, 5, 2, 7, 3]
[3, 5, 1, 2, 7, 9, 8, 6, 4]
[8, 7, 2, 6, 3, 4, 9, 5, 1]
[1, 6, 7, 9, 8, 3, 5, 4, 2]
[5, 3, 8, 4, 2, 7, 6, 1, 9]
[4, 2, 9, 5, 6, 1, 7, 3, 8]
[2, 1, 5, 3, 9, 6, 4, 8, 7]
[6, 9, 3, 7, 4, 8, 1, 2, 5

In [29]:
for i in range(len(boards_medium)):
    for j in range(len(boards_medium)):
        runTests(boards_medium[j])

[1, 6, 2, 7, 4, 9, 8, 5, 3]
[5, 7, 4, 3, 8, 2, 1, 9, 6]
[8, 9, 3, 5, 6, 1, 7, 2, 4]
[9, 4, 7, 8, 2, 6, 5, 3, 1]
[3, 5, 1, 9, 7, 4, 2, 6, 8]
[6, 2, 8, 1, 3, 5, 4, 7, 9]
[4, 1, 5, 6, 9, 7, 3, 8, 2]
[7, 8, 9, 2, 1, 3, 6, 4, 5]
[2, 3, 6, 4, 5, 8, 9, 1, 7]
Valid solution: True, {'Nodes': 53, 'Backtracks': 4} in 0.0127 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[5, 6, 8, 7, 4, 9, 2, 1, 3]
[3, 1, 9, 5, 2, 6, 7, 4, 8]
[7, 4, 2, 3, 8, 1, 6, 9, 5]
[9, 8, 1, 6, 5, 2, 4, 3, 7]
[2, 5, 4, 9, 3, 7, 1, 8, 6]
[6, 7, 3, 4, 1, 8, 9, 5, 2]
[1, 9, 7, 8, 6, 3, 5, 2, 4]
[4, 3, 6, 2, 9, 5, 8, 7, 1]
[8, 2, 5, 1, 7, 4, 3, 6, 9]
Valid solution: True, {'Nodes': 52, 'Backtracks': 2} in 0.0113 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[2, 7, 4, 8, 1, 6, 3, 9, 5]
[3, 9, 5, 7, 2, 4, 8, 6, 1]
[8, 1, 6, 9, 3, 5, 7, 2, 4]
[6, 4, 8, 3, 7, 1, 9, 5, 2]
[7, 5, 3, 6, 9, 2, 1, 4, 8]
[1, 2, 9, 5, 4, 8, 6, 7, 3]
[4, 3, 2, 1, 6, 9, 5, 8, 7]
[5, 6, 7, 4, 8, 3, 2, 1, 9

In [30]:
for i in range(len(boards_hard)):
    for j in range(len(boards_hard)):
        runTests(boards_hard[j])

[9, 1, 7, 6, 3, 2, 8, 4, 5]
[6, 4, 2, 5, 8, 7, 1, 9, 3]
[8, 3, 5, 9, 1, 4, 2, 7, 6]
[5, 6, 3, 4, 7, 1, 9, 8, 2]
[7, 2, 9, 8, 6, 5, 3, 1, 4]
[1, 8, 4, 2, 9, 3, 6, 5, 7]
[4, 9, 8, 7, 2, 6, 5, 3, 1]
[3, 7, 6, 1, 5, 9, 4, 2, 8]
[2, 5, 1, 3, 4, 8, 7, 6, 9]
Valid solution: True, {'Nodes': 55, 'Backtracks': 3} in 0.0137 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[4, 7, 1, 3, 9, 5, 2, 6, 8]
[2, 9, 5, 8, 7, 6, 4, 3, 1]
[3, 8, 6, 1, 4, 2, 7, 9, 5]
[1, 4, 8, 9, 2, 7, 6, 5, 3]
[6, 2, 3, 4, 5, 8, 1, 7, 9]
[7, 5, 9, 6, 1, 3, 8, 4, 2]
[9, 3, 2, 7, 8, 4, 5, 1, 6]
[5, 1, 7, 2, 6, 9, 3, 8, 4]
[8, 6, 4, 5, 3, 1, 9, 2, 7]
Valid solution: True, {'Nodes': 97, 'Backtracks': 55} in 0.0268 seconds 
 MRV = True, Forward Checking = True, Arc Consistnecy 3 = True 

[7, 5, 8, 4, 2, 6, 1, 3, 9]
[2, 9, 3, 5, 1, 7, 8, 6, 4]
[6, 1, 4, 8, 3, 9, 5, 7, 2]
[8, 4, 2, 6, 7, 3, 9, 5, 1]
[5, 7, 1, 9, 4, 8, 3, 2, 6]
[9, 3, 6, 2, 5, 1, 4, 8, 7]
[1, 8, 9, 3, 6, 2, 7, 4, 5]
[3, 2, 5, 7, 9, 4, 6, 1, 